# CMC R2 — Full 768-token cross-model sensitivity (Gemma + Qwen)

Formal 768-token sensitivity run after the Qwen n=50 calibration passed the pre-specified truncation gate.

**Design**
- Models: `google/gemma-2-9b-it` and `Qwen/Qwen2.5-7B-Instruct`
- Same frozen 500 detector-state target edges per dataset (1,500/model)
- Methods: Basic Prompt vs Proposed Source-Gated Prompt
- 3,000 generations/model, 6,000 total
- `max_new_tokens=768`, BF16, SDPA, batch size 4, `do_sample=False`
- Each model uses its own published `generation_config.eos_token_id`
- Main 384 runs are preserved; 768 outputs go to new Drive roots
- After generation: exact field metrics, paired statistics, and AlignScore are recomputed


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Clone current repository

In [ ]:
import os, shutil, stat, subprocess, sys, json, platform
from pathlib import Path
from google.colab import userdata

REPO_URL='https://github.com/ramita99/cmc_llm_87708.git'
REPO_DIR=Path('/content/cmc_llm_768_full')
github_token=userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Colab Secrets에 GITHUB_TOKEN을 추가하세요.')
askpass=Path('/content/git_askpass_cmc768.sh')
askpass_script='''#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
'''
askpass.write_text(askpass_script,encoding='utf-8')
askpass.chmod(askpass.stat().st_mode|stat.S_IXUSR)
env=os.environ.copy(); env['GITHUB_TOKEN']=github_token; env['GIT_ASKPASS']=str(askpass); env['GIT_TERMINAL_PROMPT']='0'
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
try:
    subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True,env=env)
finally:
    askpass.unlink(missing_ok=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src=str(REPO_DIR/'src')
if src not in sys.path: sys.path.insert(0,src)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Repo HEAD:',HEAD)


## 2. GPU and Hugging Face login

In [ ]:
import torch, transformers, yaml, pandas as pd
from huggingface_hub import login, snapshot_download, model_info, hf_hub_download

if not torch.cuda.is_available(): raise RuntimeError('GPU runtime required')
props=torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GiB | BF16: {torch.cuda.is_bf16_supported()}')
if not torch.cuda.is_bf16_supported(): raise RuntimeError('BF16-capable GPU required')
hf_token=userdata.get('HF_TOKEN')
if hf_token: login(token=hf_token,add_to_git_credential=False)


## 3. Reuse the same frozen 500 detector-state packets per dataset

In [ ]:
from cmc87708.protocol import assert_llm_facing_packet, stable_hash

SOURCE_ROOT=Path('/content/drive/MyDrive/CMC_R2_full_frozen_20260918')
GEMMA_ROOT=Path('/content/drive/MyDrive/CMC_R2_gemma2_full500_768_20260919')
QWEN_ROOT=Path('/content/drive/MyDrive/CMC_R2_qwen25_full500_768_20260919')
DATASETS=['NF-CSE-CIC-IDS2018-v3','NF-ToN-IoT-v3','NF-UNSW-NB15-v3']

def load_jsonl(path):
    with Path(path).open('r',encoding='utf-8') as f:
        return [json.loads(x) for x in f if x.strip()]

for root in [GEMMA_ROOT,QWEN_ROOT]:
    for sub in ['packets','manifests','results']:
        (root/sub).mkdir(parents=True,exist_ok=True)
    manifest=[]
    for ds in DATASETS:
        source=[r for r in load_jsonl(SOURCE_ROOT/'packets'/f'{ds}.jsonl') if r.get('state')=='detector']
        assert len(source)==500,(ds,len(source))
        with (root/'packets'/f'{ds}.jsonl').open('w',encoding='utf-8') as f:
            for p in source:
                assert_llm_facing_packet(p,'768 sensitivity packet')
                f.write(json.dumps(p,ensure_ascii=False,default=str)+'\n')
                t=p['traceability']
                manifest.append({'dataset':ds,'window_id':t.get('window_id'),'edge_id':t.get('edge_id'),'packet_hash':stable_hash(p)})
    m=pd.DataFrame(manifest)
    assert len(m)==1500 and m.groupby('dataset').size().eq(500).all()
    m.to_csv(root/'manifests'/'full500_768_target_manifest.csv',index=False)
print('Copied the same 1,500 detector-state packets into both 768 runs.')


## 4. Pin model revisions and write configs

In [ ]:
MODELS={
 'gemma2_9b_it': {'id':'google/gemma-2-9b-it','root':GEMMA_ROOT},
 'qwen25_7b_instruct': {'id':'Qwen/Qwen2.5-7B-Instruct','root':QWEN_ROOT},
}

for label,spec in MODELS.items():
    info=model_info(spec['id'],token=hf_token)
    spec['revision']=info.sha
    spec['local']=snapshot_download(repo_id=spec['id'],revision=spec['revision'],token=hf_token)
    cfg={
      'protocol_version':f'CMC-R2-768-{label}-2026-09-19',
      'source_main_protocol':'CMC-R2-FROZEN-2026-09-18',
      'output_root':str(spec['root']),
      'sample_seed':7,
      'targets_per_dataset':500,
      'sampling':'same_frozen_500_targets_as_main_run',
      'states':['detector'],
      'methods':['basic_prompt','proposed_source_gated'],
      'full_state_for_method_comparison':'detector',
      'model_id':spec['id'],
      'model_revision':spec['revision'],
      'generation_dtype':'bfloat16',
      'attention_implementation':'sdpa',
      'generation_batch_size':4,
      'max_new_tokens':768,
      'do_sample':False,
      'numeric_tolerance':0.0001,
      'fallback_on_truncation':True,
      'bootstrap_replicates':10000,
      'multiple_testing_correction':'holm',
      'expected_datasets':DATASETS,
      'expected_targets_per_dataset':500,
      'expected_packet_states_per_target':1,
    }
    cfg_path=Path(f'/content/{label}_768.yaml')
    cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')
    spec['config']=cfg_path
    (spec['root']/f'{label}_768.yaml').write_text(yaml.safe_dump(cfg,sort_keys=False),encoding='utf-8')
    envrec={
      'model_id':spec['id'],'model_revision':spec['revision'],'repo_head':HEAD,
      'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
      'transformers':transformers.__version__,'python':platform.python_version(),
      'max_new_tokens':768,'paired_targets':1500,'expected_generations':3000
    }
    (spec['root']/'execution_environment.json').write_text(json.dumps(envrec,indent=2),encoding='utf-8')
    print(label,spec['revision'])


## 5A. Run Gemma 768 full

This cell is resume-safe. If Colab disconnects, rerun the same cell.

In [ ]:
G_MODEL=str(MODELS['gemma2_9b_it']['local'])
G_CFG=str(MODELS['gemma2_9b_it']['config'])
!python scripts/02_run_primary_generation.py --experiment {G_CFG} --model {G_MODEL} --methods basic_prompt proposed_source_gated --resume


## 5B. Verify Gemma 3,000/3,000 before moving on

In [ ]:
def count_jsonl(path):
    if not Path(path).exists(): return 0
    with Path(path).open('r',encoding='utf-8') as f: return sum(1 for x in f if x.strip())

def verify_root(root):
    total=0
    for method in ['basic_prompt','proposed_source_gated']:
        for ds in DATASETS:
            n=count_jsonl(root/'outputs'/method/f'{ds}.jsonl')
            print(f'{method:24s} {ds:24s} {n}/500')
            assert n==500,(method,ds,n)
            total+=n
    assert total==3000
    print('Complete:',total)

verify_root(GEMMA_ROOT)


## 6A. Run Qwen 768 full

This cell is resume-safe. The n=50 calibration already showed 0% truncation in all six dataset × method cells.

In [ ]:
Q_MODEL=str(MODELS['qwen25_7b_instruct']['local'])
Q_CFG=str(MODELS['qwen25_7b_instruct']['config'])
!python scripts/02_run_primary_generation.py --experiment {Q_CFG} --model {Q_MODEL} --methods basic_prompt proposed_source_gated --resume


## 6B. Verify Qwen 3,000/3,000

In [ ]:
verify_root(QWEN_ROOT)


## 7. Exact-field evaluation + native/final paired statistics

In [ ]:
for label,spec in MODELS.items():
    cfg=str(spec['config'])
    print('\n===',label,'===')
    subprocess.run([sys.executable,'scripts/03_recompute_primary_metrics.py','--experiment',cfg],check=True)
    subprocess.run([sys.executable,'scripts/04_primary_paired_statistics.py','--experiment',cfg],check=True)


## 8. Summarize full-sample truncation/fallback before AlignScore

In [ ]:
for label,spec in MODELS.items():
    p=spec['root']/'results'/'summary_metrics_audit_v2.csv'
    df=pd.read_csv(p)
    keep=df[(df['state']=='detector') & df['method'].isin(['basic_prompt','proposed_source_gated'])]
    cols=[c for c in ['dataset','method','generated_tokens','terminated_by_eos','truncated_at_max_tokens','audit_pass_generation','fallback_used','native_required_field_completeness'] if c in keep.columns]
    print('\n===',label,'768 summary ===')
    display(keep[cols])


## 9. AlignScore for both 768 runs

In [ ]:
import nltk
nltk.download('punkt',quiet=True)
try: nltk.download('punkt_tab',quiet=True)
except Exception: pass
CKPT=hf_hub_download(repo_id='yzha/AlignScore',filename='AlignScore-large.ckpt',token=hf_token)

for label,spec in MODELS.items():
    out_label=label+'_768'
    subprocess.run([
      sys.executable,'scripts/05_primary_alignscore.py',
      '--root',str(spec['root']),'--label',out_label,
      '--checkpoint',CKPT,'--batch-size','32','--device','cuda',
      '--bootstrap-replicates','10000','--seed','7'
    ],check=True)


## 10. Combined 768 summary

In [ ]:
summaries=[]; stats=[]
for label,spec in MODELS.items():
    out_label=label+'_768'
    summaries.append(pd.read_csv(spec['root']/'results'/f'alignscore_summary_{out_label}.csv'))
    stats.append(pd.read_csv(spec['root']/'results'/f'alignscore_paired_statistics_{out_label}.csv'))
combined_summary=pd.concat(summaries,ignore_index=True)
combined_stats=pd.concat(stats,ignore_index=True)
display(combined_summary)
display(combined_stats)
OUT=Path('/content/drive/MyDrive/CMC_R2_cross_model_768_combined_20260919')
OUT.mkdir(parents=True,exist_ok=True)
combined_summary.to_csv(OUT/'alignscore_768_combined_summary.csv',index=False)
combined_stats.to_csv(OUT/'alignscore_768_combined_paired_statistics.csv',index=False)
print('Combined 768 results:',OUT)
